In [0]:
%sql
USE CATALOG gold_dev;
USE SCHEMA analytics;

In [0]:
%sql
CREATE OR REPLACE VIEW vw_sales_base_masked AS
SELECT
  f.order_id,
  od.date AS order_date,
  sd.date AS ship_date,
  DATEDIFF(sd.date, od.date) AS days_to_ship,
  -- Customer with masked name
  c.customer_key,
  c.customer_id,
  -- Masked name: first letter + stars
  CONCAT(
    SUBSTRING(c.customer_name, 1, 1), REPEAT('*', LENGTH(c.customer_name) - 1)
  ) AS customer_name,
  c.customer_segment,
  -- Region
  r.region AS region,
  -- Product
  p.product_key,
  p.product_id,
  p.product_name,
  p.category,
  p.sub_category,
  f.order_quantity,
  f.sales_amount,
  f.discount_amount,
  f.profit_amount,
  f.ingestion_ts,
  f.load_timestamp
FROM
  silver_dev.global_mart_retail.fact_sales f
    JOIN silver_dev.global_mart_retail.dim_customer c
      ON f.customer_key = c.customer_key
      AND c.is_current_record = true
    JOIN silver_dev.global_mart_retail.dim_product p
      ON f.product_key = p.product_key
      AND p.is_current_record = true
    JOIN silver_dev.global_mart_retail.dim_date od
      ON f.order_date_key = od.date_key
    JOIN silver_dev.global_mart_retail.dim_date sd
      ON f.ship_date_key = sd.date_key
    LEFT JOIN silver_dev.global_mart_retail.dim_region r
      ON f.region_key = r.region_key